# 31 — Afectaciones que declaran las personas cuidadoras por cuidar (ENASIC 2022)

**Para qué decisión existe este notebook.** Decidir si el Observatorio muestra el costo en salud del cuidado, con qué universo y en qué gráfica, y dejar cifras que se puedan rastrear a un archivo en vez de una tabla copiada a mano.

**Salidas.** `resultados/31_afectaciones_cuidadoras.csv` (por sexo y universo; control contra la presentación de INEGI) y `resultados/31_afectaciones_por_horas.csv` (por horas de cuidado y por segmento; es lo que muestra el Observatorio en «¿Quién lo carga?»).

**Alcance.** Nacional. La ENASIC no permite estimaciones por alcaldía ni para la CDMX; nada de aquí se presenta como dato local.
Es **autorreporte**: la pregunta es «por el cuidado que usted otorga, ¿…?». No es diagnóstico ni prueba que el cuidado cause la afectación.

**Datos.** `TPOB_CUI` (personas cuidadoras de 15 años o más), factor `FAC_CUI`. Las mismas siete preguntas están en dos bloques del cuestionario:

| Bloque | Quién lo contesta | Variables |
|---|---|---|
| 6.11 | cuidadoras de personas con discapacidad o dependencia (`FILTRO6_1 = 1`) | `P6_11_1` … `P6_11_7` |
| 6.30 | cuidadoras de las otras subpoblaciones (`FILTRO6_2, 3, 6, 8`) | `P6_30_1` … `P6_30_7` |

Regla de traslape: quien contestó los dos bloques cuenta «sí» si dijo «sí» en cualquiera (misma regla que el script 27, función `cualq`). Esta regla reproduce las cifras de la presentación de INEGI (sección 4); contar solo el primer bloque no las reproduce.

**Dos universos.**
- **A · Todas las personas cuidadoras** (`FILTRO6_1, 2, 3, 6` u `8`): el universo de la presentación de INEGI.
- **B · Quienes cuidan a una persona con discapacidad o de 60+** (`FILTRO6_1` o `FILTRO6_6`): el universo del resto del Observatorio (pestaña «¿Quién lo carga?», scripts 27 a 30).

**Método.** Igual que el notebook B, sección 1: cociente de totales ponderados, varianza por linealización de Taylor con estratos `EST_DIS` y conglomerados `UPM_DIS`, IC al 90 %. La diferencia mujeres − hombres se linealiza junta, porque comparten conglomerados.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
pd.set_option('display.width', 180); pd.set_option('display.max_colwidth', 70)

BASE = Path('datos/enasic')
OUT = Path('resultados'); OUT.mkdir(exist_ok=True)
Z90 = 1.6449

cui = pd.read_csv(BASE / 'conjunto_de_datos_tpob_cui_enasic_2022' / 'conjunto_de_datos' / 'conjunto_de_datos_tpob_cui_enasic_2022.csv', dtype=str)
cui.columns = [c.upper() for c in cui.columns]
cui['W'] = pd.to_numeric(cui.FAC_CUI)
print(f'TPOB_CUI: {len(cui)} registros · estratos={cui.EST_DIS.nunique()} · UPM={cui.UPM_DIS.nunique()} · Σ FAC_CUI={cui.W.sum():,.0f}')
print('SEXO:', cui.SEXO.value_counts().to_dict(), '(catálogo INEGI: 1 = Hombre, 2 = Mujer)')

TPOB_CUI: 5677 registros · estratos=148 · UPM=861 · Σ FAC_CUI=31,652,127
SEXO: {'2': 4173, '1': 1504} (catálogo INEGI: 1 = Hombre, 2 = Mujer)


## 1. Variables y universos

`ITEMS` lista las siete afectaciones con su par de columnas (bloque 6.11, bloque 6.30). `SALUD3` es el desenlace principal del modelo de focalización (DEC-OBJ-002): salud física deteriorada, cansancio o menos sueño, con la misma definición que el script 27.

In [2]:
ITEMS = [  # clave, texto de la pregunta, columna 6.11, columna 6.30
    ('cansado',  'Se siente cansado/a',                                            'P6_11_3', 'P6_30_3'),
    ('sueno',    'Ha disminuido su tiempo de sueño',                               'P6_11_6', 'P6_30_6'),
    ('irritado', 'Se siente irritado/a',                                           'P6_11_5', 'P6_30_5'),
    ('deprimido','Se siente deprimido/a',                                          'P6_11_4', 'P6_30_4'),
    ('salud',    'Se ha deteriorado su salud física',                              'P6_11_1', 'P6_30_1'),
    ('terapia',  'Ha recibido terapia por ansiedad, angustia, nervios o depresión','P6_11_2', 'P6_30_2'),
    ('enfermedad','Ha desarrollado alguna enfermedad o se le ha agravado',         'P6_11_7', 'P6_30_7'),
]
def si(a, b): return ((cui[a] == '1') | (cui[b] == '1')).to_numpy(int)
def respondio(a, b): return (cui[a].isin(['1', '2']) | cui[b].isin(['1', '2'])).to_numpy()

Y = {k: si(a, b) for k, _, a, b in ITEMS}
# Desenlace principal del modelo: salud física, cansancio o sueño (idéntico al script 27)
Y['salud3'] = np.maximum.reduce([Y['salud'], Y['cansado'], Y['sueno']])
TEXTO = {k: t for k, t, _, _ in ITEMS} | {'salud3': 'Al menos una de: salud física deteriorada, cansancio o menos sueño'}

RESP = respondio('P6_11_1', 'P6_30_1')
F = lambda c: (cui[c] == '1').to_numpy()
UNIVERSOS = {
    'A · Todas las personas cuidadoras': (F('FILTRO6_1') | F('FILTRO6_2') | F('FILTRO6_3') | F('FILTRO6_6') | F('FILTRO6_8')) & RESP,
    'B · Cuidan a una persona con discapacidad o de 60+': (F('FILTRO6_1') | F('FILTRO6_6')) & RESP,
}
for u, m in UNIVERSOS.items():
    print(f'{u:55s} n={m.sum():5d}  Σ FAC_CUI={cui.W[m].sum():>12,.0f}')
# Control: ¿quién es cuidadora según INEGI pero no contestó el bloque?
cuid_inegi = F('FILTRO6_1') | F('FILTRO6_2') | F('FILTRO6_3') | F('FILTRO6_6') | F('FILTRO6_8')
print('cuidadoras según FILTRO6_x:', cuid_inegi.sum(), '· sin respuesta en 6.11 ni 6.30:', (cuid_inegi & ~RESP).sum())

A · Todas las personas cuidadoras                       n= 5031  Σ FAC_CUI=  28,363,859
B · Cuidan a una persona con discapacidad o de 60+      n= 1043  Σ FAC_CUI=   6,223,684
cuidadoras según FILTRO6_x: 5031 · sin respuesta en 6.11 ni 6.30: 0


## 2. Función de estimación

La misma de la sección 1 del notebook B, más la diferencia mujeres − hombres. Estratos con una sola UPM no aportan a la varianza.

In [3]:
def _puntajes(dom, y):
    """Puntajes linealizados de la proporción Σw·y·dom / Σw·dom, por registro."""
    w = cui.W.to_numpy(float); x = dom.astype(float); y = np.asarray(y, float)
    X = (w * x).sum(); R = (w * y * x).sum() / X
    return R, w * x * (y - R) / X

def _var(z):
    t = pd.DataFrame({'h': cui.EST_DIS.values, 'u': cui.UPM_DIS.values, 'z': z}).groupby(['h', 'u']).z.sum().reset_index()
    g = t.groupby('h').z; n_h = g.transform('size'); zbar = g.transform('mean')
    return float(((n_h / (n_h - 1)).where(n_h > 1, 0) * (t.z - zbar) ** 2).sum())

def estimar(dom, y):
    R, z = _puntajes(dom, y); se = np.sqrt(_var(z)); cv = se / R * 100 if R > 0 else np.nan
    prec = 'alta' if cv < 15 else ('moderada' if cv < 30 else 'baja')
    return dict(pct=R * 100, se=se * 100, cv=cv, li90=(R - Z90 * se) * 100, ls90=(R + Z90 * se) * 100, precision=prec,
                n_muestra_dominio=int(dom.sum()), n_muestra_si=int((dom & (np.asarray(y) == 1)).sum()),
                total_expandido=float((cui.W.to_numpy(float) * dom * np.asarray(y)).sum()))

def diferencia(dom_a, dom_b, y):
    """a − b en puntos porcentuales; los dos dominios comparten conglomerados, así que se linealizan juntos."""
    Ra, za = _puntajes(dom_a, y); Rb, zb = _puntajes(dom_b, y); se = np.sqrt(_var(za - zb))
    d = (Ra - Rb) * 100; li, ls = d - Z90 * se * 100, d + Z90 * se * 100
    return dict(dif_pp=d, dif_li90=li, dif_ls90=ls, dif_significativa=bool(li > 0 or ls < 0))

## 3. Estimaciones por universo, sexo e indicador

Una fila por universo × sexo × indicador. En las filas de mujeres se agrega la diferencia mujeres − hombres con su IC al 90 %.

In [4]:
MUJER, HOMBRE = (cui.SEXO == '2').to_numpy(), (cui.SEXO == '1').to_numpy()
filas = []
for u, base in UNIVERSOS.items():
    for k, y in Y.items():
        for sexo, ms in [('Total', np.ones(len(cui), bool)), ('Mujeres', MUJER), ('Hombres', HOMBRE)]:
            r = estimar(base & ms, y[:]) | dict(universo=u, sexo=sexo, clave=k, indicador=TEXTO[k])
            if sexo == 'Mujeres': r |= diferencia(base & MUJER, base & HOMBRE, y)
            filas.append(r)
res = pd.DataFrame(filas)[['universo', 'sexo', 'clave', 'indicador', 'pct', 'cv', 'li90', 'ls90', 'n_muestra_dominio', 'n_muestra_si',
                           'total_expandido', 'precision', 'dif_pp', 'dif_li90', 'dif_ls90', 'dif_significativa']]
res['fuente'] = 'INEGI, ENASIC 2022, TPOB_CUI (P6.11 y P6.30), FAC_CUI'
res.round(2).query("clave in ['cansado','sueno','irritado','deprimido','salud','terapia','enfermedad','salud3'] and sexo != 'Total'").pivot_table(
    index=['universo', 'clave'], columns='sexo', values='pct').round(1)

sexo                                                           Hombres  Mujeres
universo                                           clave                       
A · Todas las personas cuidadoras                  cansado        15.2     39.1
                                                   deprimido       6.9     16.3
                                                   enfermedad      2.5      5.3
                                                   irritado        7.4     22.7
                                                   salud           6.4     12.7
                                                   salud3         24.0     49.5
                                                   sueno          17.3     31.7
                                                   terapia         3.0      6.4
B · Cuidan a una persona con discapacidad o de 60+ cansado        27.1     46.5
                                                   deprimido      14.7     29.6
                                                   enfermedad      5.4     11.9
                                                   irritado       12.3     29.8
                                                   salud          14.8     27.4
                                                   salud3         34.0     55.4
                                                   sueno          23.6     36.8
                                                   terapia         6.3     10.0

## 4. Control contra la presentación de INEGI

Cifras de la presentación de resultados de la ENASIC 2022 («Impacto en la salud de las personas cuidadoras»), **transcritas a mano por Omar el 3 nov 2026; la lámina exacta está por confirmar**. Si no coinciden con el universo A al decimal, el notebook se detiene: significaría que cambió la regla de traslape o el universo.

In [5]:
INEGI = {  # clave: (hombres %, mujeres %)
    'cansado': (15.2, 39.1), 'sueno': (17.3, 31.7), 'irritado': (7.4, 22.7), 'deprimido': (6.9, 16.3),
    'salud': (6.4, 12.7), 'terapia': (3.0, 6.4), 'enfermedad': (2.5, 5.3)}
A = res[res.universo.str.startswith('A')].set_index(['clave', 'sexo']).pct
ctrl = pd.DataFrame([{'clave': k, 'INEGI H': h, 'calculado H': round(A[(k, 'Hombres')], 1), 'INEGI M': m, 'calculado M': round(A[(k, 'Mujeres')], 1)}
                     for k, (h, m) in INEGI.items()])
ctrl['coincide'] = (ctrl['INEGI H'] == ctrl['calculado H']) & (ctrl['INEGI M'] == ctrl['calculado M'])
display(ctrl)
assert ctrl.coincide.all(), 'El universo A ya no reproduce la presentación de INEGI: revisar universo y regla de traslape'
print('Las 14 cifras de la presentación de INEGI se reproducen al decimal con el universo A y la regla «sí en cualquiera de los dos bloques».')

,clave,INEGI H,calculado H,INEGI M,calculado M,coincide
0,cansado,15.2,15.2,39.1,39.1,True
1,sueno,17.3,17.3,31.7,31.7,True
2,irritado,7.4,7.4,22.7,22.7,True
3,deprimido,6.9,6.9,16.3,16.3,True
4,salud,6.4,6.4,12.7,12.7,True
5,terapia,3.0,3.0,6.4,6.4,True
6,enfermedad,2.5,2.5,5.3,5.3,True


Las 14 cifras de la presentación de INEGI se reproducen al decimal con el universo A y la regla «sí en cualquiera de los dos bloques».


## 5. Qué cambia con el universo

El universo B (quienes cuidan a una persona con discapacidad o de 60+) declara **más** afectación que el universo A en las 16 comparaciones (8 indicadores × 2 sexos). Para el Observatorio, que trabaja con el universo B, usar las cifras de INEGI (universo A) **subestimaría** el costo en salud de su propio universo.

In [6]:
b = res[(res.clave.isin(['cansado', 'sueno', 'irritado', 'deprimido', 'salud', 'terapia', 'enfermedad', 'salud3'])) & (res.sexo != 'Total')]
cmp_ = b.pivot_table(index=['clave', 'sexo'], columns=b.universo.str[0], values='pct').round(1)
cmp_['B − A (pp)'] = (cmp_['B'] - cmp_['A']).round(1)
cmp_

universo               A     B  B − A (pp)
clave      sexo                           
cansado    Hombres  15.2  27.1        11.9
           Mujeres  39.1  46.5         7.4
deprimido  Hombres   6.9  14.7         7.8
           Mujeres  16.3  29.6        13.3
enfermedad Hombres   2.5   5.4         2.9
           Mujeres   5.3  11.9         6.6
irritado   Hombres   7.4  12.3         4.9
           Mujeres  22.7  29.8         7.1
salud      Hombres   6.4  14.8         8.4
           Mujeres  12.7  27.4        14.7
salud3     Hombres  24.0  34.0        10.0
           Mujeres  49.5  55.4         5.9
sueno      Hombres  17.3  23.6         6.3
           Mujeres  31.7  36.8         5.1
terapia    Hombres   3.0   6.3         3.3
           Mujeres   6.4  10.0         3.6

In [7]:
dif = res[(res.sexo == 'Mujeres') & (res.clave.isin(['cansado', 'sueno', 'irritado', 'deprimido', 'salud', 'terapia', 'enfermedad', 'salud3']))]
dif[['universo', 'clave', 'dif_pp', 'dif_li90', 'dif_ls90', 'dif_significativa']].round(1)

,universo,clave,dif_pp,dif_li90,dif_ls90,dif_significativa
1,A · Todas las personas cuidadoras,cansado,23.9,20.9,26.9,True
4,A · Todas las personas cuidadoras,sueno,14.4,11.4,17.4,True
7,A · Todas las personas cuidadoras,irritado,15.3,13.2,17.3,True
10,A · Todas las personas cuidadoras,deprimido,9.4,6.9,11.8,True
13,A · Todas las personas cuidadoras,salud,6.2,4.4,8.1,True
16,A · Todas las personas cuidadoras,terapia,3.4,1.9,4.8,True
19,A · Todas las personas cuidadoras,enfermedad,2.8,1.5,4.1,True
22,A · Todas las personas cuidadoras,salud3,25.5,22.3,28.8,True
25,B · Cuidan a una persona con discapacidad o de 60+,cansado,19.5,10.9,28.0,True
28,B · Cuidan a una persona con discapacidad o de 60+,sueno,13.3,6.1,20.5,True


## 6. Lo que estas cifras no dicen

- **No son diagnóstico.** «Se siente cansada(o)» por el cuidado es percepción de quien contesta, no una medición clínica.
- **No prueban causalidad.** La encuesta no compara con personas que no cuidan. La literatura clínica sobre cuidadoras se revisa aparte; aquí no se cita ninguna.
- **Hombres: muestra pequeña.** Con pocos hombres en el universo B, algunas filas tienen precisión baja o moderada (ver la columna `precision`); el Observatorio marca con rayado las de precisión baja.
- **Nacional.** Ninguna de estas cifras sirve para hablar de una alcaldía.
- **Distinto de «algún daño» (nb. B, sección 12).** Aquel indicador incluye también afectaciones laborales y usa otra definición de universo; no son intercambiables.

## 7. Por horas de cuidado: lo que muestra el Observatorio

La tesis de la pestaña «¿Quién lo carga?» es que la carga crece con las horas. Aquí cada afectación se estima en tres grupos de horas (menos de 20, 20 a 39, 40 o más) en el universo B, para el total y para cada segmento del filtro de la pestaña (los mismos de la sección 12b del notebook B: sexo, edad, ingreso y a quién cuida). Con el filtro de Intensidad no se calcula: ya está cortado por horas.

Definiciones copiadas de la sección 16 del notebook B: `HORAS` = P6.7 + P6.26 con tope de 168 (vacío si faltan las dos), quintiles de ingreso del hogar con los cortes del script 04. Las filas de 40 h o más llevan la diferencia contra menos de 20 h; esa diferencia es para el equipo (va en «Fuente y precisión»), no para el texto que lee el usuario.

In [8]:
num = lambda c: pd.to_numeric(cui[c], errors='coerce')
cui['HORAS'] = (num('P6_7').fillna(0) + num('P6_26').fillna(0)).clip(upper=168)
cui.loc[num('P6_7').isna() & num('P6_26').isna(), 'HORAS'] = np.nan
cui['G_HORAS'] = pd.cut(cui.HORAS, [0, 19.999, 39.999, 1000], labels=['<20 h', '20-39 h', '>=40 h']).astype(str)

hog = pd.read_csv(BASE / 'conjunto_de_datos_thogar_enasic_2022' / 'conjunto_de_datos' / 'conjunto_de_datos_thogar_ensasic_2022.csv', dtype=str)  # typo del INEGI
hog.columns = [c.upper() for c in hog.columns]
MID = {'1': 1800, '2': 4850.5, '3': 7100.5, '4': 9000.5, '5': 11000.5, '6': 13300.5, '7': 16050.5, '8': 19750.5, '9': 25500.5, '10': 37100.5, '11': 55000}
hog['ING'] = pd.to_numeric(hog.P3A_4.where(hog.P3A_4 != '99999'), errors='coerce').fillna(hog.P3A_5.map(MID))
nec = ((hog.HN_CDISC == '1') | (hog.HN_C60MA == '1')) & hog.ING.notna(); W_h = pd.to_numeric(hog.FAC_HOG)
def wq(v, w, q):
    o = np.argsort(v.values); c = np.cumsum(w.values[o]) / w.sum(); return v.values[o][np.searchsorted(c, q)]
cortes = [wq(hog.ING[nec], W_h[nec], q) for q in (.2, .4, .6, .8)]
assert cortes == [3200, 5000, 8000, 15000], f'Los cortes de quintil no coinciden con el script 04: {cortes}'
hog['QUINTIL'] = np.where(hog.ING.isna(), np.nan, np.digitize(hog.ING, cortes, right=True) + 1)
cui = cui.merge(hog[['LLAVEHOG', 'QUINTIL']], on='LLAVEHOG', how='left', validate='many_to_one')

cui['G_SEXO'] = cui.SEXO.map({'1': 'Hombre', '2': 'Mujer'})
cui['G_EDAD'] = pd.cut(num('EDAD'), [0, 39, 59, 130], labels=['15-39', '40-59', '60+']).astype(str)
cui['G_ING'] = cui.QUINTIL.map({1: 'Q1-Q2', 2: 'Q1-Q2', 3: 'Q3', 4: 'Q4-Q5', 5: 'Q4-Q5'})
cui['G_TIPO'] = np.where(cui.FILTRO6_1 == '1', 'Cuida discapacidad', 'Solo 60+ sin discapacidad')
SEG = [('Total', 'Total', None)]
SEG += [('Sexo', g, 'G_SEXO') for g in ['Mujer', 'Hombre']]
SEG += [('Edad', g, 'G_EDAD') for g in ['15-39', '40-59', '60+']]
SEG += [('Ingreso', g, 'G_ING') for g in ['Q1-Q2', 'Q3', 'Q4-Q5']]
SEG += [('A quién cuida', g, 'G_TIPO') for g in ['Cuida discapacidad', 'Solo 60+ sin discapacidad']]

UB = UNIVERSOS['B · Cuidan a una persona con discapacidad o de 60+']
HB = {g: (cui.G_HORAS == g).to_numpy() for g in ['<20 h', '20-39 h', '>=40 h']}
print('Universo B por horas:', {g: int((UB & m).sum()) for g, m in HB.items()}, '· sin horas:', int((UB & cui.HORAS.isna().to_numpy()).sum()))

Universo B por horas: {'<20 h': 374, '20-39 h': 300, '>=40 h': 369} · sin horas: 0


In [9]:
fh = []
for dim, g, col in SEG:
    ms = np.ones(len(cui), bool) if col is None else (cui[col] == g).to_numpy()
    for k, y in Y.items():
        for h, mh in HB.items():
            r = estimar(UB & ms & mh, y) | dict(dimension=dim, segmento=g, clave=k, indicador=TEXTO[k], horas=h)
            if h == '>=40 h': r |= diferencia(UB & ms & HB['>=40 h'], UB & ms & HB['<20 h'], y)
            fh.append(r)
porh = pd.DataFrame(fh)[['dimension', 'segmento', 'clave', 'indicador', 'horas', 'pct', 'cv', 'li90', 'ls90', 'n_muestra_dominio', 'n_muestra_si',
                         'total_expandido', 'precision', 'dif_pp', 'dif_li90', 'dif_ls90', 'dif_significativa']]
porh['fuente'] = 'INEGI, ENASIC 2022, TPOB_CUI (P6.11 y P6.30), FAC_CUI; universo B'
porh[porh.dimension == 'Total'].pivot_table(index='clave', columns='horas', values='pct').round(1)[['<20 h', '20-39 h', '>=40 h']]

horas,<20 h,20-39 h,>=40 h
clave,,,
cansado,28.8,39.5,50.2
deprimido,17.5,23.6,31.7
enfermedad,8.5,7.1,12.8
irritado,22.0,20.5,28.7
salud,14.1,20.4,32.8
salud3,35.8,45.7,60.8
sueno,22.4,29.6,42.8
terapia,6.7,6.8,11.8


In [10]:
t = porh[(porh.dimension == 'Total') & (porh.horas == '>=40 h')][['clave', 'dif_pp', 'dif_li90', 'dif_ls90', 'dif_significativa']].round(1)
print('40 h o más frente a menos de 20 h (total, universo B):'); print(t.to_string(index=False))
print('Celdas con precisión baja por dimensión:'); print(porh[porh.precision == 'baja'].groupby('dimension').size())

40 h o más frente a menos de 20 h (total, universo B):
     clave  dif_pp  dif_li90  dif_ls90 dif_significativa
   cansado    21.4      13.6      29.2              True
     sueno    20.4      12.7      28.2              True
  irritado     6.7      -0.4      13.8             False
 deprimido    14.2       8.2      20.2              True
     salud    18.7      11.1      26.4              True
   terapia     5.1       0.3      10.0              True
enfermedad     4.3      -1.5      10.1             False
    salud3    25.1      16.3      33.9              True
Celdas con precisión baja por dimensión:
dimension
A quién cuida    12
Edad             26
Ingreso          23
Sexo             14
Total             1
dtype: int64


In [11]:
res.to_csv(OUT / '31_afectaciones_cuidadoras.csv', index=False, encoding='utf-8-sig')
porh.to_csv(OUT / '31_afectaciones_por_horas.csv', index=False, encoding='utf-8-sig')
for f in ['31_afectaciones_cuidadoras.csv', '31_afectaciones_por_horas.csv']:
    chk = pd.read_csv(OUT / f, encoding='utf-8-sig'); print(f, chk.shape)
assert len(porh) == 11 * 8 * 3, 'Se esperaban 11 segmentos × 8 indicadores × 3 grupos de horas'

31_afectaciones_cuidadoras.csv (48, 17)
31_afectaciones_por_horas.csv (264, 18)
